# v16b — does training on hard negatives teach the instruction/injection boundary?

v15's finding (b): the detector fires on a **benign imperative inside untrusted content** at
40.75–52.39%, against 19–20% on the untouched carrier and 60–71% on real injections. It appears
to detect *instruction presence*, not *maliciousness*.

But it was never trained on a benign instruction. v16 tests whether that is a property of the
representation or an artifact of the training set.

## Four experiments

| # | Question | Decides |
|---|---|---|
| 1 | **Arm A vs Arm B** — hard negatives held out of training vs included | does showing the boundary teach it, and what does it cost? |
| 2 | **Nested per-context thresholds** | the honest headline ASR, properly nested this time |
| 3 | **Injection vs hard negative, directly** | can the representation separate them *at all*? the sharpest form of finding (b) |
| 4 | **Cost and complexity** | owed since 2026-09-27 |

**Exp 3 is the one that matters scientifically.** Both classes contain an imperative sentence
inside untrusted content at the same position, length-matched; only one is an attack. If a
classifier trained on exactly that contrast cannot separate them, finding (b) is a property of
the latent space and a strong claim. If it can, then v14's detector was simply never asked to.

## Two corrections carried in from v15

- v14's honest operating point (**ASR 7.09%**) had thresholds fitted on the rows they were
  measured on. Exp 2 nests them properly — calibration half, measurement half, resampled.
- Exp 2 in v15 was mis-specified because the chunk cache held only two benign contexts. Here the
  full eight-corpus benign suite is present, so the threshold-spread question is answerable.

In [1]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore"); sns.set_style("whitegrid")

SEED, N_THRESH, FPR_BUDGET = 42, 400, 0.10
TARGET_ASR, TARGET_FPR = 5.0, 10.0
PCA_DIMS, C_REG, N_SPLITS = 512, 10.0, 5

def make_logreg(): return LogisticRegression(class_weight="balanced", max_iter=5000,
                                             C=C_REG, solver="lbfgs", random_state=SEED)
def make_hgb():    return HistGradientBoostingClassifier(max_iter=300, random_state=SEED)
def make_rf():     return RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                 class_weight="balanced_subsample",
                                                 random_state=SEED, n_jobs=-1)
def make_mlp():    return MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=600,
                                        random_state=SEED)
CLF = {"LogisticRegression": make_logreg, "HistGradientBoost": make_hgb,
       "RandomForest": make_rf, "MLP": make_mlp}
PRIMARY = "HistGradientBoost"          # v14 run 2's finding: the head is what mattered

CACHE = "representation_cache/joined_v16"
RESULT_DIR, FIG = "results/v16", "mslsamv16"
os.makedirs(RESULT_DIR, exist_ok=True); os.makedirs("outputs", exist_ok=True)

man = json.load(open(f"{CACHE}/manifest.json")); LAYERS = man["layers"]
df  = pd.read_csv(f"{CACHE}/rows_v16.csv").reset_index(drop=True)
assert man["rows"] == len(df)
print(df.groupby(["corpus","label"]).size().unstack(fill_value=0).to_string())
print(f"\n{len(df):,} rows | {df.label.sum():,} inj | {(df.label==0).sum():,} ben"
      f" | groups {df.group_carrier.nunique():,} (carrier) / {df.group_component.nunique():,} (component)")
print(f"sources: {man['sources']}")

label                  0     1
corpus                        
BIPIA                  0  1200
BIPIA-Clean          480     0
BIPIA-HardNeg        481     0
Baseline-Dolly      2000     0
DeepSet                0   263
DeepSet-Benign       300     0
InjecAgent             0   186
InjecAgent-Clean     479     0
InjecAgent-HardNeg   509     0
JBB-Benign           100     0
SPML                   0  2499
SPML-Benign         1198     0
WJB-Adv-Benign      2500     0
WJB-Vanilla-Benign   800     0
XSTest-Safe          250     0

13,245 rows | 4,148 inj | 9,097 ben | groups 8,042 (carrier) / 7,596 (component)
sources: {'v14': 9910, 'v15_last_chunk': 3335}


In [2]:
# ---- PCA on the Dolly baseline, then v14's feature set ----------------------
is_base = (df.corpus == "Baseline-Dolly").values
ev = df[~is_base].reset_index(drop=True)
ev_rows = np.where(~is_base)[0]

PC, MU, CINV = {}, {}, {}
for li in LAYERS:
    X = np.load(f"{CACHE}/layer_{li}.npy")
    N_PCA = min(PCA_DIMS, int(is_base.sum()) - 1, X.shape[1])
    p = PCA(n_components=N_PCA).fit(X[is_base])
    Z = p.transform(X).astype(np.float32)
    PC[li] = Z[ev_rows]
    b = p.transform(X[is_base])
    MU[li] = b.mean(0)
    CINV[li] = np.linalg.inv(np.cov(b.T) + np.eye(N_PCA) * 1e-4)
    del X, Z, b
print(f"PCA {N_PCA} comps/layer on {int(is_base.sum()):,} baseline rows")

def _norm(A): return A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)

def max_cos_exact(q_rows, p_rows, li):
    # exact self-match masking: remove the self COLUMN before the max, never compare
    # against it. (v11 got this wrong positionally; v15c got it wrong by comparison.)
    S = _norm(PC[li][q_rows]) @ _norm(PC[li][p_rows]).T
    pos = {int(r): j for j, r in enumerate(p_rows)}
    for i_, r in enumerate(q_rows):
        j = pos.get(int(r))
        if j is not None: S[i_, j] = -np.inf
    return S.max(axis=1)

def mahal(Z, li):
    d = Z - MU[li]
    return np.sqrt(np.einsum("ij,jk,ik->i", d, CINV[li], d))

FAMILIES = ["direct", "indirect"]

def features(rows, proto):
    # v14's feature set, unchanged: prototype max-cosine per family x layer, plus
    # Mahalanobis per layer. 8 layers x (2 + 1) = 24 features.
    cols, names = [], []
    for fam in FAMILIES:
        pr = proto.get(fam, np.array([], int))
        for li in LAYERS:
            cols.append(np.zeros(len(rows), np.float32) if len(pr) == 0
                        else max_cos_exact(rows, pr, li))
            names.append(f"P_{fam}_L{li}")
    for li in LAYERS:
        cols.append(mahal(PC[li][rows], li)); names.append(f"M_L{li}")
    return np.column_stack(cols).astype(np.float32), names

PCA 512 comps/layer on 2,000 baseline rows


In [3]:
# ---- protocol: nested thresholds, component-grouped folds ------------------
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; den = 1 + z*z/n; c = (p + z*z/(2*n))/den
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return max(0, c-h)*100, min(1, c+h)*100

def nested(inj, ben, budget=FPR_BUDGET, n_rep=N_THRESH, seed=0, extra=None):
    rng = np.random.default_rng(seed); inj, ben = np.asarray(inj), np.asarray(ben)
    if len(ben) < 2 or len(inj) == 0: return None
    half = max(1, len(ben)//2); ts, fp, asr = [], [], []
    ex = {k: [] for k in (extra or {})}
    for _ in range(n_rep):
        i = rng.permutation(len(ben)); t = float(np.quantile(ben[i[:half]], 1-budget))
        ts.append(t); fp.append((ben[i[half:]] >= t).mean()*100); asr.append((inj < t).mean()*100)
        for k, v in (extra or {}).items():
            ex[k].append((np.asarray(v) >= t).mean()*100 if len(v) else np.nan)
    q = lambda a: [float(np.mean(a)), float(np.percentile(a,2.5)), float(np.percentile(a,97.5))]
    y = np.r_[np.zeros(len(ben)), np.ones(len(inj))]
    lo, hi = wilson(int(round(np.mean(asr)/100*len(inj))), len(inj))
    return {"auc": float(roc_auc_score(y, np.r_[ben, inj])),
            "ap": float(average_precision_score(y, np.r_[ben, inj])),
            "threshold": q(ts), "fpr": q(fp), "asr": q(asr), "asr_binom": [lo, hi],
            "cross": {k: q(v) for k, v in ex.items()}, "n_inj": len(inj), "n_ben": len(ben)}

fmt = lambda v: f"{v[0]:5.2f} [{v[1]:5.2f},{v[2]:5.2f}]"

GROUP_KEY = "group_carrier"     # see v16a: component grouping puts ALL InjecAgent rows in
                                # one fold, which would silently make Exp 1 leave-one-family-out
                                # for InjecAgent and break comparability with v14.

def grouped_cv(train_mask, clf_name=PRIMARY, seed=SEED, score_extra=None, verbose=False,
               group_key=None):
    # Folds are grouped so a carrier's injections and its hard-negative TWIN can never
    # straddle the split -- otherwise arm B could win by recognising the document.
    rows = np.where(train_mask)[0]
    y = ev.label.values[rows]
    g = ev[group_key or GROUP_KEY].values[rows]
    oof = np.full(len(rows), np.nan)
    ex = None if score_extra is None else np.zeros((N_SPLITS, len(score_extra)))
    skf = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=seed)
    for k, (tr, te) in enumerate(skf.split(np.zeros(len(rows)), y, groups=g)):
        tr_rows, te_rows = rows[tr], rows[te]
        proto = {f: tr_rows[(y[tr] == 1) & (ev.family.values[tr_rows] == f)] for f in FAMILIES}
        Xtr, nmz = features(tr_rows, proto)
        Xte, _   = features(te_rows, proto)
        sc = RobustScaler().fit(Xtr)
        c = CLF[clf_name]().fit(sc.transform(Xtr), y[tr])
        oof[te] = c.predict_proba(sc.transform(Xte))[:, 1]
        if ex is not None:
            Xx, _ = features(np.asarray(score_extra), proto)
            ex[k] = c.predict_proba(sc.transform(Xx))[:, 1]
        if verbose:
            print(f"    fold {k}: train {len(tr):5d} test {len(te):5d} "
                  f"| test inj {int(y[te].sum()):4d} ben {int((y[te]==0).sum()):4d}")
    return rows, y, oof, (None if ex is None else ex.mean(0))

INJ_C  = ["BIPIA", "InjecAgent", "SPML", "DeepSet"]
HARD_C = ["BIPIA-HardNeg", "InjecAgent-HardNeg"]
hard_mask = ev.corpus.isin(HARD_C).values
inj_mask  = (ev.label.values == 1)
print(f"grouping: {GROUP_KEY} -> {ev[GROUP_KEY].nunique():,} groups "
      f"(component key kept for Exp 3 sensitivity)")
print("corpora:", sorted(ev.corpus.unique()))
print(f"hard negatives: {int(hard_mask.sum()):,}")

grouping: group_carrier -> 6,042 groups (component key kept for Exp 3 sensitivity)
corpora: ['BIPIA', 'BIPIA-Clean', 'BIPIA-HardNeg', 'DeepSet', 'DeepSet-Benign', 'InjecAgent', 'InjecAgent-Clean', 'InjecAgent-HardNeg', 'JBB-Benign', 'SPML', 'SPML-Benign', 'WJB-Adv-Benign', 'WJB-Vanilla-Benign', 'XSTest-Safe']
hard negatives: 990


In [4]:
# ================= EXP 1 — Arm A vs Arm B ====================================
# A: hard negatives held OUT of training, scored by the fold models (this is v14's detector,
#    measured against a control it never saw -- reproduces v15's finding (b)).
# B: hard negatives INCLUDED in training as label 0.
E1 = {}
hard_rows = np.where(hard_mask)[0]

print("ARM A - hard negatives held out of training")
rowsA, yA, oofA, sA = grouped_cv(~hard_mask, score_extra=hard_rows, verbose=True)
print("\nARM B - hard negatives in training")
rowsB, yB, oofB, _   = grouped_cv(np.ones(len(ev), bool), verbose=True)

def summarise(rows, y, oof, extra_scores=None, extra_rows=None):
    ben = oof[y == 0]; inj = oof[y == 1]
    cross = {c: oof[(y == 0) & (ev.corpus.values[rows] == c)]
             for c in sorted(set(ev.corpus.values[rows][y == 0]))}
    if extra_scores is not None:
        for c in HARD_C:
            cross[c] = extra_scores[ev.corpus.values[extra_rows] == c]
    pt = nested(inj, ben, extra=cross)
    pt["detection_by_corpus"] = {
        c: float((oof[(y == 1) & (ev.corpus.values[rows] == c)] >= pt["threshold"][0]).mean()*100)
        for c in INJ_C if ((y == 1) & (ev.corpus.values[rows] == c)).any()}
    return pt

A = summarise(rowsA, yA, oofA, sA, hard_rows)
B = summarise(rowsB, yB, oofB)
E1 = {"armA_heldout": A, "armB_trained": B}

print(f"\n  {'':<22}{'ARM A':>26}{'ARM B':>26}")
print("  " + "-"*74)
print(f"  {'ROC-AUC':<22}{A['auc']:>26.4f}{B['auc']:>26.4f}")
print(f"  {'AP':<22}{A['ap']:>26.4f}{B['ap']:>26.4f}")
print(f"  {'ASR %':<22}{fmt(A['asr']):>26}{fmt(B['asr']):>26}")
wA = f"[{A['asr_binom'][0]:.2f},{A['asr_binom'][1]:.2f}]"
wB = f"[{B['asr_binom'][0]:.2f},{B['asr_binom'][1]:.2f}]"
print(f"  {'ASR % Wilson':<22}{wA:>26}{wB:>26}")
print(f"  {'pooled FPR %':<22}{fmt(A['fpr']):>26}{fmt(B['fpr']):>26}")
print(f"\n  {'benign corpus':<24}{'ARM A FPR %':>14}{'ARM B FPR %':>14}{'delta':>9}")
print("  " + "-"*61)
for c in sorted(set(list(A['cross']) + list(B['cross']))):
    a = A['cross'].get(c, [np.nan]*3)[0]; b = B['cross'].get(c, [np.nan]*3)[0]
    star = "  <- hard neg" if c in HARD_C else ""
    print(f"  {c:<24}{a:>13.2f}%{b:>13.2f}%{b-a:>+8.2f}{star}")
print(f"\n  {'attack corpus':<24}{'ARM A det %':>14}{'ARM B det %':>14}{'delta':>9}")
print("  " + "-"*61)
for c in INJ_C:
    a = A['detection_by_corpus'].get(c, np.nan); b = B['detection_by_corpus'].get(c, np.nan)
    print(f"  {c:<24}{a:>13.2f}%{b:>13.2f}%{b-a:>+8.2f}")
print("\n  v15 reference (finding b): BIPIA-HardNeg 40.75-52.39% FPR, never trained on.")

ARM A - hard negatives held out of training
    fold 0: train  8156 test  2099 | test inj  844 ben 1255
    fold 1: train  8211 test  2044 | test inj  833 ben 1211
    fold 2: train  8143 test  2112 | test inj  890 ben 1222
    fold 3: train  8246 test  2009 | test inj  804 ben 1205
    fold 4: train  8264 test  1991 | test inj  777 ben 1214

ARM B - hard negatives in training
    fold 0: train  9076 test  2169 | test inj  800 ben 1369
    fold 1: train  8933 test  2312 | test inj  851 ben 1461
    fold 2: train  8995 test  2250 | test inj  843 ben 1407
    fold 3: train  8941 test  2304 | test inj  836 ben 1468
    fold 4: train  9035 test  2210 | test inj  818 ben 1392

                                             ARM A                     ARM B
  --------------------------------------------------------------------------
  ROC-AUC                                   0.9820                    0.9726
  AP                                        0.9747                    0.9593
  ASR %    

In [5]:
# ================= EXP 2 — nested per-context thresholds =====================
# v14's honest 7.09% had thresholds fitted on the rows they were measured on. Here each
# context's threshold is set on a calibration half and measured on the held-out half, 400x.
PAIR = {"BIPIA": "BIPIA-Clean", "InjecAgent": "InjecAgent-Clean",
        "SPML": "SPML-Benign", "DeepSet": "DeepSet-Benign"}
GLOBAL_BEN = ["WJB-Adv-Benign", "WJB-Vanilla-Benign", "XSTest-Safe", "JBB-Benign"]

E2 = {}
for arm, (rows, y, oof) in {"armA_heldout": (rowsA, yA, oofA),
                            "armB_trained": (rowsB, yB, oofB)}.items():
    cor = ev.corpus.values[rows]
    res, ti, tm = {}, 0, 0.0
    for ic, bc in PAIR.items():
        si = oof[(y == 1) & (cor == ic)]
        sb = oof[(y == 0) & (cor == bc)]
        if not len(si) or len(sb) < 2: continue
        extra = {g: oof[(y == 0) & (cor == g)] for g in GLOBAL_BEN}
        for h in HARD_C:
            if ic.split("-")[0] in h: extra[h] = oof[(y == 0) & (cor == h)]
        pt = nested(si, sb, extra=extra)
        if pt is None: continue
        res[ic] = pt; ti += len(si); tm += pt["asr"][0]/100*len(si)
    res["_pooled_asr"] = float(tm/ti*100) if ti else float("nan")
    res["_n_inj"] = int(ti)
    E2[arm] = res

for arm, res in E2.items():
    print(f"\n=== {arm} ===")
    print(f"  {'family':<12}{'threshold':>11}{'ASR %':>20}{'own-ctx FPR %':>16}")
    print("  " + "-"*59)
    for ic in PAIR:
        if ic not in res: continue
        p = res[ic]
        print(f"  {ic:<12}{p['threshold'][0]:>11.6f}{fmt(p['asr']):>20}{p['fpr'][0]:>15.2f}%")
    print("  " + "-"*59)
    print(f"  POOLED context-matched ASR: {res['_pooled_asr']:.2f}%  (n={res['_n_inj']:,})")
print("\n  v14 reference: 7.09% -- but with thresholds fitted on the measured rows.")
print("  These are nested, so they are the first defensible version of that number.")

print("\n=== threshold spread across ALL benign contexts (v15's Exp 2, done properly) ===")
for arm, (rows, y, oof) in {"armA_heldout": (rowsA, yA, oofA),
                            "armB_trained": (rowsB, yB, oofB)}.items():
    cor = ev.corpus.values[rows]
    th = {c: float(np.quantile(oof[(y == 0) & (cor == c)], 0.90))
          for c in sorted(set(cor[y == 0]))}
    lo, hi = min(th.values()), max(th.values())
    E2[arm]["_spread"] = {"thresholds": th, "spread": hi/max(lo, 1e-12)}
    print(f"  {arm}: spread {hi/max(lo,1e-12):.3g}x over {len(th)} contexts")
    for c, t in sorted(th.items(), key=lambda x: -x[1]): print(f"      {c:<24}{t:.6f}")
print("  v14 run 2: 7.5e3x over 8 contexts.")


=== armA_heldout ===
  family        threshold               ASR %   own-ctx FPR %
  -----------------------------------------------------------
  BIPIA          0.971377 31.26 [24.24,35.92]          10.52%
  InjecAgent     0.702137 47.06 [45.16,48.92]          10.10%
  SPML           0.006127  0.91 [ 0.76, 1.08]           9.91%
  DeepSet        0.011848  8.07 [ 5.32,12.55]          10.76%
  -----------------------------------------------------------
  POOLED context-matched ASR: 12.21%  (n=4,148)

=== armB_trained ===
  family        threshold               ASR %   own-ctx FPR %
  -----------------------------------------------------------
  BIPIA          0.851432 36.60 [30.75,42.59]          10.29%
  InjecAgent     0.161255 69.10 [65.05,70.97]          10.48%
  SPML           0.010540  0.89 [ 0.80, 1.08]           9.95%
  DeepSet        0.021910  8.79 [ 6.08,12.93]          10.93%
  -----------------------------------------------------------
  POOLED context-matched ASR: 14.78%  (n

In [6]:
# ================= EXP 3 — injection vs hard negative, directly =============
# THE experiment. Both classes carry an imperative sentence inside untrusted content, at the
# same position, length-matched. Only one is an attack. If a classifier trained on exactly
# this contrast cannot separate them, v15's finding (b) is a property of the latent space.
# Grouping is reported under BOTH keys, because neither is clean on its own:
#   group_carrier   closes carrier recognition; a payload can recur across carriers
#   group_component closes both, but is unusable for InjecAgent (one component)
# If the two agree, the conclusion is robust to the choice. If they disagree, the gap IS
# the uncertainty and must be reported as such.
E3 = {}
for fam, hard in [("BIPIA", "BIPIA-HardNeg"), ("InjecAgent", "InjecAgent-HardNeg")]:
    m = ev.corpus.isin([fam, hard]).values
    rows = np.where(m)[0]
    y = (ev.corpus.values[rows] == fam).astype(int)      # 1 = real injection
    E3[fam] = {}
    for gk in ["group_carrier", "group_component"]:
        g = ev[gk].values[rows]
        ng = len(set(g)); frac = pd.Series(g).value_counts().iloc[0]/len(g)
        if ng < N_SPLITS or frac > 0.5:
            print(f"\n--- {fam} vs {hard} [{gk}] SKIPPED: {ng} groups, largest "
                  f"{frac*100:.0f}% of rows -- cannot make {N_SPLITS} folds")
            continue
        print(f"\n--- {fam} vs {hard} [{gk}] --- n_inj {int(y.sum())} "
              f"n_hard {int((y==0).sum())} | {ng} groups, largest {frac*100:.0f}%")
        E3[fam][gk] = {}
        for name in CLF:
            oof = np.full(len(rows), np.nan)
            skf = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=SEED)
            for tr, te in skf.split(np.zeros(len(rows)), y, groups=g):
                tr_rows, te_rows = rows[tr], rows[te]
                proto = {f: tr_rows[(y[tr] == 1) & (ev.family.values[tr_rows] == f)]
                         for f in FAMILIES}
                Xtr, _ = features(tr_rows, proto); Xte, _ = features(te_rows, proto)
                sc = RobustScaler().fit(Xtr)
                c = CLF[name]().fit(sc.transform(Xtr), y[tr])
                oof[te] = c.predict_proba(sc.transform(Xte))[:, 1]
            pt = nested(oof[y == 1], oof[y == 0])
            E3[fam][gk][name] = pt
            print(f"    {name:<20} AUC {pt['auc']:.4f}  AP {pt['ap']:.4f}  "
                  f"ASR {fmt(pt['asr'])}  FPR {pt['fpr'][0]:5.2f}%")
        aucs = [v["auc"] for v in E3[fam][gk].values()]
        print(f"    best AUC {max(aucs):.4f} | classifier spread {max(aucs)-min(aucs):.4f}")

print("\n=== Exp 3 verdict ===")
for fam in E3:
    if not E3[fam]: print(f"  {fam}: no usable grouping"); continue
    best = {gk: max(v["auc"] for v in d.values()) for gk, d in E3[fam].items()}
    lo, hi = min(best.values()), max(best.values())
    print(f"  {fam:<11} best AUC {lo:.4f}-{hi:.4f} across groupings {list(best)}")
    if hi < 0.70:
        print(f"    -> NO classifier separates a malicious instruction from a benign one.")
        print(f"       Finding (b) is a property of the REPRESENTATION, not of v14's")
        print(f"       training set. That is a strong negative claim and the thesis result.")
    elif lo > 0.85:
        print(f"    -> separable when asked directly: v14's detector simply never was asked.")
    else:
        print(f"    -> partially separable, and grouping-sensitive. Report the RANGE, not a")
        print(f"       point estimate, and treat the gap as the uncertainty.")


--- BIPIA vs BIPIA-HardNeg [group_carrier] --- n_inj 1200 n_hard 481 | 481 groups, largest 1%
    LogisticRegression   AUC 0.7791  AP 0.8839  ASR 60.49 [53.41,66.34]  FPR 10.27%
    HistGradientBoost    AUC 0.8108  AP 0.9084  ASR 50.77 [47.08,56.50]  FPR 10.42%
    RandomForest         AUC 0.8256  AP 0.9218  ASR 47.44 [43.58,50.67]  FPR 10.20%
    MLP                  AUC 0.8179  AP 0.9145  ASR 44.90 [40.33,52.42]  FPR 10.58%
    best AUC 0.8256 | classifier spread 0.0465

--- BIPIA vs BIPIA-HardNeg [group_component] --- n_inj 1200 n_hard 481 | 51 groups, largest 19%
    LogisticRegression   AUC 0.7384  AP 0.8590  ASR 67.76 [59.67,73.42]  FPR 10.27%
    HistGradientBoost    AUC 0.7815  AP 0.8992  ASR 55.81 [51.17,60.33]  FPR 10.18%
    RandomForest         AUC 0.7817  AP 0.9002  ASR 57.53 [54.42,60.50]  FPR 10.40%
    MLP                  AUC 0.7984  AP 0.9079  ASR 50.35 [45.42,57.33]  FPR 10.66%
    best AUC 0.7984 | classifier spread 0.0600

--- InjecAgent vs InjecAgent-HardNeg [gro

In [7]:
# ================= EXP 4 — cost and complexity ==============================
# Owed since 2026-09-27. The forward pass dominates and must be measured, not asserted;
# its number comes from v15b's own extraction log on this machine.
E4 = {}
proto_all = {f: np.where(inj_mask & (ev.family.values == f))[0] for f in FAMILIES}
n_bench = min(2000, len(ev))
bench_rows = np.random.default_rng(SEED).choice(len(ev), n_bench, replace=False)

t0 = time.perf_counter(); Xb, nmz = features(bench_rows, proto_all); t_feat = time.perf_counter()-t0
sc = RobustScaler().fit(Xb)
clf = CLF[PRIMARY]().fit(sc.transform(Xb), ev.label.values[bench_rows])
Xs = sc.transform(Xb)
t0 = time.perf_counter()
for _ in range(5): clf.predict_proba(Xs)
t_inf = (time.perf_counter()-t0)/5

n_proto = sum(len(v) for v in proto_all.values())
E4 = {"n_rows": int(n_bench), "n_features": int(Xb.shape[1]), "n_prototypes": int(n_proto),
      "feature_ms_per_row": t_feat/n_bench*1e3, "inference_ms_per_row": t_inf/n_bench*1e3,
      "pca_dims": int(N_PCA), "layers": len(LAYERS)}
print(f"  feature construction : {E4['feature_ms_per_row']:.3f} ms/prompt "
      f"({n_proto:,} prototypes, {len(LAYERS)} layers, {N_PCA} PCA dims)")
print(f"  classifier inference : {E4['inference_ms_per_row']:.4f} ms/prompt")
print(f"  detector total       : {E4['feature_ms_per_row']+E4['inference_ms_per_row']:.3f} ms/prompt")
print(f"\n  Llama-3-8B forward pass (the dominant term), measured by v15b on this machine:")
print(f"    ~0.5 rows/s batched on MPS = ~2000 ms/prompt at MAX_LEN=512")
print(f"    -> the detector adds ~{(E4['feature_ms_per_row']+E4['inference_ms_per_row'])/2000*100:.2f}% "
      f"on top of the pass it already needs")
print(f"\n  ASYMPTOTICS (per prompt, n = prompt tokens, d = 4096, k = {N_PCA}):")
print(f"    forward pass          O(n^2 d + n d^2)   <- dominates; one prefill")
print(f"    PCA projection        O(L d k)           L = {len(LAYERS)} layers")
print(f"    prototype max-cosine  O(L k P)           P = {n_proto:,} prototypes  <- grows with training set")
print(f"    Mahalanobis           O(L k^2)")
print(f"    HistGradientBoost     O(trees x depth)   effectively constant")
print(f"\n  The honest framing: the guard costs ONE PREFILL of the prompt it is protecting,")
print(f"  plus <1 ms. An LLM-judge baseline costs a full generation. That is the comparison.")
print(f"  Caveat: prototype max-cosine is linear in the training set, so P must be capped")
print(f"  (or prototypes clustered) before this scales to a production corpus.")

  feature construction : 3.212 ms/prompt (4,148 prototypes, 8 layers, 512 PCA dims)
  classifier inference : 0.0048 ms/prompt
  detector total       : 3.217 ms/prompt

  Llama-3-8B forward pass (the dominant term), measured by v15b on this machine:
    ~0.5 rows/s batched on MPS = ~2000 ms/prompt at MAX_LEN=512
    -> the detector adds ~0.16% on top of the pass it already needs

  ASYMPTOTICS (per prompt, n = prompt tokens, d = 4096, k = 512):
    forward pass          O(n^2 d + n d^2)   <- dominates; one prefill
    PCA projection        O(L d k)           L = 8 layers
    prototype max-cosine  O(L k P)           P = 4,148 prototypes  <- grows with training set
    Mahalanobis           O(L k^2)
    HistGradientBoost     O(trees x depth)   effectively constant

  The honest framing: the guard costs ONE PREFILL of the prompt it is protecting,
  plus <1 ms. An LLM-judge baseline costs a full generation. That is the comparison.
  Caveat: prototype max-cosine is linear in the training set

In [8]:
# ---- persist + verdict -----------------------------------------------------
strip = lambda d: {k: v for k, v in d.items() if k not in ("oof", "rows")}
out = dict(version="v16", run_at=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
           pca_components=int(N_PCA), seed=SEED, classifier=PRIMARY,
           rows=int(len(ev)), groups=int(ev[GROUP_KEY].nunique()), group_key_used=GROUP_KEY,
           group_key=man["group_key"], sources=man["sources"],
           exp1_arms={k: strip(v) for k, v in E1.items()},
           exp2_context_matched=E2, exp3_inj_vs_hardneg=E3, exp4_cost=E4)
json.dump(out, open(f"{RESULT_DIR}/metrics.json", "w"), indent=2, default=float)
pd.DataFrame({"score": oofB, "label": yB, "corpus": ev.corpus.values[rowsB],
              "group": ev[GROUP_KEY].values[rowsB],
              "pair_id": ev.pair_id.values[rowsB]}).to_csv(f"{RESULT_DIR}/oof_v16.csv", index=False)
print(f"wrote {RESULT_DIR}/metrics.json + oof_v16.csv")

print("\n--- VERDICT ---")
hn_A = np.mean([E1['armA_heldout']['cross'][c][0] for c in HARD_C if c in E1['armA_heldout']['cross']])
hn_B = np.mean([E1['armB_trained']['cross'][c][0] for c in HARD_C if c in E1['armB_trained']['cross']])
print(f"  hard-negative FPR: arm A {hn_A:.2f}%  ->  arm B {hn_B:.2f}%  ({hn_B-hn_A:+.2f} pts)")
print(f"  context-matched ASR (nested): A {E2['armA_heldout']['_pooled_asr']:.2f}%"
      f"  ->  B {E2['armB_trained']['_pooled_asr']:.2f}%")
best3 = {f: (max(max(d["auc"] for d in gd.values()) for gd in E3[f].values())
                 if E3[f] else float("nan")) for f in E3}
print(f"  Exp 3 best AUC (inj vs hard-neg): {{{', '.join(f'{k}: {v:.4f}' for k,v in best3.items())}}}")
worst = max((v[0] for c, v in E1['armB_trained']['cross'].items() if c not in HARD_C), default=np.nan)
hit = (E2['armB_trained']['_pooled_asr'] < TARGET_ASR) and (worst < TARGET_FPR)
print(f"  arm B worst non-hardneg benign FPR {worst:.2f}%")
print(f"  TARGET (ASR<{TARGET_ASR}, all benign FPR<{TARGET_FPR}): {'MET' if hit else 'not met'}")
print("\nUpdate versions/15_mslsam-v16/README.md, EXPERIMENTS.md, AGENTS.md (R2/R3/R5).")

wrote results/v16/metrics.json + oof_v16.csv

--- VERDICT ---
  hard-negative FPR: arm A 50.65%  ->  arm B 32.74%  (-17.91 pts)
  context-matched ASR (nested): A 12.21%  ->  B 14.78%
  Exp 3 best AUC (inj vs hard-neg): {BIPIA: 0.8256, InjecAgent: 0.8116}
  arm B worst non-hardneg benign FPR 34.24%
  TARGET (ASR<5.0, all benign FPR<10.0): not met

Update versions/15_mslsam-v16/README.md, EXPERIMENTS.md, AGENTS.md (R2/R3/R5).
